## Trajectory Inspection, Parsing, and Action Validation
Before passing raw model text into OpenEnv or computing rewards, an agent system must robustly parse and validate the LLM's outputs.

## 1. Action Extraction: Text-to-Structured-Action Pipeline

The LLM outputs a raw token sequence. OpenEnv uses typed models such as Pydantic models or typed dataclasses to validate that the action conforms to the environment's required action schema.

### Example flow

```text
LLM Raw Output String:
"I will now inspect the test file.\n```bash\ncat tests/test_runner.py\n```"

        │
        ▼
(Action Parser / Regex / JSON Extractor)

Extracted Action Dict:
{"name": "bash", "command": "cat tests/test_runner.py"}

        │
        ▼
(Pydantic Schema Validation)

Validated Action Instance:
BashAction(command="cat tests/test_runner.py")

        │
        ▼
Passed to env.step(action)
```

## 2. Handling Malformed Actions (The Action Validation Failure Loop)

What happens if the model hallucinates an invalid JSON payload, forgets closing markdown backticks, or calls a tool that does not exist?

In agent RL, there are two distinct ways to handle syntax or validation failures:

```text
LLM Emits Malformed Action
        │
┌───────┴───────────────────────┐
│                               │
▼                               ▼
Option A: Fatal Abort          Option B: Feedback Loop
(Hard Failure)                 (Environment Error)
• Episode ends immediately    • Step count increments (t = t + 1)
• Reward = 0.0 or -1.0         • Obs = "SyntaxError: Malformed JSON"
• terminated = True            • Agent gets another chance to correct syntax
```

### Option A: Fatal Abort (Hard Failure)

- Punishes formatting mistakes severely.
- Used when training base models to learn strict format adherence during initial warmups.

### Option B: Feedback Loop (Environment Error)

- Treats the syntax parser as part of the environment.
- The error message is appended to the trajectory, allowing the LLM to learn self-correction over multi-step interactions.

## Question

In a multi-turn OpenEnv setup where the environment runs in Docker:

If an agent generates an action that triggers an infinite loop (for example, `python -c "while True: pass"`), how should the OpenEnv execution runner handle this so it does not freeze the entire distributed training worker?

### Answer

In production systems such as OpenEnv's Docker runner or pytest harnesses, the correct engineering approach is a three-part mechanism:

1. **Subprocess timeout**: wrap the shell command in a strict wall-clock timeout such as 5.0 seconds.
2. **Signal group termination**: if the timeout expires, terminate the entire process group with `SIGKILL` so child processes cannot survive as CPU-hungry zombies.
3. **Synthetic timeout observation**: return a controlled observation instead of crashing the GPU training loop.

```json
{
  "status": "timeout",
  "exit_code": 124,
  "output": "Command timed out after 5.0s. Execution aborted."
}
```

### Why this works

- The episode can continue safely without freezing the distributed worker.
- The LLM receives the timeout error and can revise its action.
- The system can apply a penalty if the agent exhausts its maximum steps.